# Train a deepwarp model

Runtime → Change runtime type → GPU, then run the cells in order.

- **Quant·ion** reads each file: only the bytes of the target traces are fetched.
- **deepwarp** trains on the files already read while the next files are being read.

The two libraries never import each other.

In [ ]:
!pip install -q "quantion>=0.2.3"
!pip install -q "deepwarp @ git+https://github.com/josoriom/deep-warp"

## 1. Your files

Paths or http URLs of `.ion` files.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import glob

folder = "/content/drive/MyDrive/samples"
files = sorted(glob.glob(f"{folder}/*.ion"))
print(len(files), "files")

## 2. Your targets

`rt` is the reference retention time in minutes. Isomers share the same `mz`.

In [ ]:
targets = [
    {"id": "Alanine", "mz": 260.103, "rt": 2.43},
    {"id": "Alanine-13C3,15N [IS]", "mz": 264.1101, "rt": 2.43},
    {"id": "Isoleucine", "mz": 302.1499, "rt": 4.13},
    {"id": "Leucine", "mz": 302.1499, "rt": 4.217},
    {"id": "Isoleucine-13C6,15N [IS]", "mz": 309.1671, "rt": 4.13},
    {"id": "Leucine-13C6,15N [IS]", "mz": 309.1671, "rt": 4.214},
]

## 3. How to read one file

Quant·ion finds the peak of each target near its `rt` (± 0.1 min) and its trace on a grid of 1024 points. A target with no peak gets `nan` and is left out of training.

In [ ]:
import numpy as np
import quantion

import deepwarp


def read_file(file, options):
    targets = options["targets"]
    grid = np.linspace(options["time_from"], options["time_to"], options["points"])
    eic = {"from": options["time_from"] - 0.05, "to": options["time_to"] + 0.05, "ppm_tolerance": options["ppm_tolerance"], "mz_tolerance": options["mz_tolerance"]}
    search = [{**target, "range": options["search_range"]} for target in targets]
    with quantion.parse_ion(file) as sample:
        peaks = quantion.find_targets(sample, search, {"eic": eic})
        traces = [quantion.calculate_eic(sample, target["mz"], eic) for target in targets]
    dataset = [np.interp(grid, trace["x"], trace["y"]) if len(trace["x"]) else np.zeros(len(grid)) for trace in traces]
    labels = [peak["rt"] if peak["intensity"] > 0 else np.nan for peak in peaks]
    return dataset, labels


## 4. Configuration 

In [ ]:
settings = deepwarp.TrainingSettings(time_from=0.5, time_to=5.0, epochs=80, uncertainty_epochs=80)
options = {
    "targets": targets,
    "time_from": settings.time_from,
    "time_to": settings.time_to,
    "points": 1024,
    "ppm_tolerance": settings.eic_ppm,
    "mz_tolerance": settings.eic_mz_tolerance,
    "search_range": 0.1,
}

## 5. Train

`deepwarp.read` reads the next 32 files in the background while the model trains on the files read so far. Each file is read once. The files are shuffled first, because the reference retention times come from the first group.

In [ ]:
random = np.random.default_rng(0)
files = list(random.permutation(files))

training = deepwarp.Training(targets, settings)
for dataset, labels in deepwarp.read(read_file, files, options, group_size=32, readers=8):
    training.add(dataset, labels)
    training.train()
model = training.finish()

deepwarp.save_model(model, f"{folder}/model.pt")
deepwarp.export_onnx(model, f"{folder}/model.onnx")

## 6. Predict with Quant·ion

In [ ]:
warp = quantion.load_warp(quantion.load_model(f"{folder}/model.onnx"))
with quantion.parse_ion(files[0]) as sample:
    predicted = warp.predict_rts(sample, targets)
for target, found in zip(targets, predicted):
    print(f'{target["id"]:<26} reference {target["rt"]:.3f}  predicted {found["rt"]:.3f} ± {found["uncertainty"]:.3f}')